<a href="https://colab.research.google.com/github/wesleylelo/CDIA_AI_COLAB/blob/develop/CROS_VALIDATION_RNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###################################################################
##################################################################
#Instituição: SENAI - CIMATEC
#Curso: Ciência de Dados e Inteligência Artificial (CDIA)
#Disciplina: Inteligência Artificial II
#Autor: Carlos Fernando Arraz | Data: Fevereiro, 2024
#UA04 - Roteiro de Prática Guiada 01: Cross validation em redes neurais
###################################################################
###################################################################

---

INSTRUÇÕES GERAIS:

---



1 - Realize uma cópia deste arquivo em seu repositório pessoal para iniciar a prática (usar o e-mail institucional para isso);

2 - Reveja o conteúdo teórico e os exemplos práticos vistos até agora na graduação para pontecializar o aprendizado;

3 -Leia a teoria e acompanhe o script python linha a linha e seus comentários (execute as células com **shift + ENTER** ou botão play à esquerda de cada comando/bloco de código), estudando as estruturas, as partes do algoritmo e a lógica proposta;

4 - Execute, modifique, teste e experimente o conteúdo ao máximo para internalizar o conhecimento; e

5- Compare os resultados com bibliotecas de machine learning consagradas no desenvolvimento de soluções em AI.

#Este roteiro tem como objetivo mostrar uma introdução sobre treinamento de uma rede neural usando a técnica *Cross Validation K-Fold*, e tudo isso através de um script em python 🐍.

**Vamos criar uma rede neural para identificar peças de vestuário de roupas distribuídas no famoso fashion MNIST, que possui 70.000 imagens e rótulos para servir de aprendizado na área de IA *(deep learning)*.**

**OBS:** A diferença entre o recurso anterior é que faremos o treinamento usando a validação cruzada *(Cross Validation K-Fold)*.

## Uso de Callbacks para controlar o treinamento

Neste laboratório, você usará a [API Callbacks](https://keras.io/api/callbacks/) para interromper o treinamento quando uma métrica especificada for atingida. Este é um recurso útil para que você não precise completar todas as épocas quando esse limite for atingido. Por exemplo, se você definir 1.000 épocas e a precisão desejada já for alcançada, por exemplo, na época 200, o treinamento será interrompido automaticamente. Vamos ver como isso é implementado nas próximas seções.


## Carregar e Normalizar o Fashion MNIST dataset

Assim como no laboratório anterior, você usará novamente o conjunto de dados Fashion MNIST para este exercício. E também como mencionado antes, você normalizará os valores dos pixels para ajudar a otimizar o treinamento.

In [ ]:
# Importar bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import confusion_matrix

import seaborn as sn

import tensorflow as tf
from tensorflow import keras

In [ ]:
# Instanciar o dataset
fmnist = tf.keras.datasets.fashion_mnist

# Carregar o dataset
(x_train, y_train),(x_test, y_test) = fmnist.load_data()

# Normalizar o valor do pixel
x_train, x_test = x_train / 255.0, x_test / 255.0

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# Verificar dimensões
x_train.shape

(60000, 28, 28)

In [ ]:
# Verificar dimensões na partição de teste
y_train.shape

(60000,)

##**[AQUI TEMOS A MUDANÇA]** Criando a classe Callback

Você pode criar um retorno de chamada definindo uma classe que herda a classe base [tf.keras.callbacks.Callback](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/Callback). A partir daí, você pode definir os métodos disponíveis para definir onde o retorno de chamada será executado. Por exemplo abaixo, você usará o método [on_epoch_end()](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/Callback#on_epoch_end) para verificar a perda em cada época de treinamento.

In [ ]:
# Criar classe myCallback
class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    '''
    Halts the training when the loss falls below 0.4

    Args:
      epoch (integer) - index of epoch (required but unused in the function definition below)
      logs (dict) - metric results from the training epoch
    '''

    # Verificar a perda (experimentar outros ajustes)
    if(logs.get('loss') < 0.4):

      # Parar se o limite estabelecido for atingido
      print("\nLoss is lower than 0.4 so cancelling training!")
      self.model.stop_training = True

# Instanciar a classe
callbacks = myCallback()

## Definr e compilar o modelo

A seguir, você definirá e compilará o modelo. A arquitetura será semelhante àquela que você construiu no recurso anterior. Depois, você definirá o otimizador, a perda e as métricas que usará para o treinamento.

In [ ]:
# Definir o modelo
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),
  tf.keras.layers.Dense(512, activation=tf.nn.relu),
  tf.keras.layers.Dense(10, activation=tf.nn.softmax)
])

# Compilar o modelo
model.compile(optimizer=tf.optimizers.Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## **[AQUI TEMOS A MUDANÇA]** Estudar em detalhes esse trecho e comparar com as práticas anteriores

Configurar Treino e Validação (K-Fold) propriedades

In [ ]:
k = 10
cross_val = KFold(k, shuffle=True, random_state=1)
fold_count = 1

# Definir quantidade de epocas
epochs = 10

# Lista para armazenar loss e acuracia
histories = []

# Lista para armazenar evolução dos scores no treino
eval_scores = []

### Treino do modelo

Agora você está pronto para treinar o modelo. Para definir o retorno de chamada, basta definir o parâmetro `callbacks` para a instância `myCallback` que você declarou antes. Execute a célula abaixo e observe o que acontece.

##**[AQUI TEMOS A MUDANÇA]** Estudar em detalhes esse trecho e comparar com as práticas anteriores

In [ ]:
# Treino do modelo com callback e CV
for train, validation in cross_val.split(X):

  print("="*80)
  print("Fold-{}".format(fold_count))
  print("-"*80)
  print("Training & Validation")
  fold_count = fold_count + 1

  # print(train, validation)

  X_train, y_train = X[train], y[train]
  X_val, y_val = X[validation], y[validation]

  history = model.fit(X_train, y_train,
                      epochs=10,
                      validation_data=(X_val, y_val),
                      callbacks=[callbacks]
                      )

  print("-"*80)
  print("Testing/evaluation")
  eval_loss, eval_accuracy = model.evaluate(x_test, y_test)

  histories.append(history)
  eval_scores.append(eval_accuracy)
  print("_"*80)

NameError: name 'X' is not defined

Você notará que o treinamento não precisa completar todas as 10 épocas. Por ter um retorno de chamada no final de cada época, ele é capaz de verificar os parâmetros de treinamento e comparar se atende ao limite definido na definição da função. Neste caso, simplesmente irá parar quando a perda cair abaixo de `0,40` após a época atual.

Isso conclui este exercício simples sobre retornos de chamada e **Cross Validation K-Fold**!

**Referências:**

[1] Código adaptado. DeepLearning.ai. Conteúdo online modificado. GitHub, https://github.com/https-deeplearning-ai/tensorflow-1-public/blob/main/C1/W2/ungraded_labs/C1_W2_Lab_1_beyond_hello_world.ipynb . Licença Apache 2.0. Acesso em: 19 fev. 2024.

[2] GÉRON, Aurélien. Mãos à Obra: Aprendizado de Máquina com Scikit-Learn, Keras & TensorFlow: Conceitos, Ferramentas e Técnicas para a Construção de Sistemas Inteligentes. Editora Alta Books, 2021. E-book. ISBN 9786555208146. Disponível em: https://integrada.minhabiblioteca.com.br/#/books/9786555208146/. Acesso em: 25 fev. 2024.

[3] FACELI, Katti; et al. Inteligência artificial: uma abordagem de aprendizado de máquina. Rio de Janeiro: Grupo GEN, 2021. E-book. ISBN 9788521637509.

[4] JAMES, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). An introduction to statistical learning: With applications in python. Springer Nature.*texto em itálico*

[5] NETTO, Amilcar; MACIEL, Francisco. Python para data science e machine learning descomplicado. Rio de Janeiro: Alta Books, 2021. E-book. ISBN 9786555203172.